In [8]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# Ensure CUDA errors are synchronous for proper debugging
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image and mask
        image = Image.open(os.path.join(self.image_dir, self.images[idx])).convert("L")
        mask = Image.open(os.path.join(self.mask_dir, self.masks[idx])).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # Normalize mask to 0-1 and ensure shape (1,H,W)
        if isinstance(mask, torch.Tensor):
            mask = mask.float()
        else:
            mask = torch.tensor(np.array(mask, dtype=np.float32) / 255.0)
        mask = mask.unsqueeze(0)

        return image.float(), mask.float()

# Transform for images
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# Dataset and DataLoader
train_dataset = BrainTumorDataset("/content/drive/MyDrive/segmentation/images", "/content/drive/MyDrive/segmentation/masks", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)  # batch size small for T4 GPU

In [9]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNetPlusPlus(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.conv1 = ConvBlock(in_ch, 64)
        self.conv2 = ConvBlock(64, 128)
        self.conv3 = ConvBlock(128, 256)
        self.conv4 = ConvBlock(256, 512)

        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.conv3d = ConvBlock(512+256, 256)
        self.conv2d = ConvBlock(256+128, 128)
        self.conv1d = ConvBlock(128+64, 64)

        self.final = nn.Conv2d(64, out_ch, 1)  # no sigmoid

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(self.pool(x1))
        x3 = self.conv3(self.pool(x2))
        x4 = self.conv4(self.pool(x3))

        x3d = self.conv3d(torch.cat([self.up(x4), x3], dim=1))
        x2d = self.conv2d(torch.cat([self.up(x3d), x2], dim=1))
        x1d = self.conv1d(torch.cat([self.up(x2d), x1], dim=1))

        out = self.final(x1d)
        return out  # logits for BCEWithLogitsLoss

In [10]:
model = UNetPlusPlus().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [15]:
from tqdm import tqdm

for epoch in range(num_epochs):
    model.train()
    running_loss = 0

    for imgs, masks in tqdm(train_loader):

        imgs = imgs.to(device)
        masks = masks.to(device)

        masks = masks.squeeze(2)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

 11%|█         | 82/766 [00:22<03:04,  3.70it/s]


KeyboardInterrupt: 

In [11]:
def dice_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()

    intersection = (pred * target).sum()
    dice = (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return dice

In [12]:
def iou_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou

In [13]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    running_dice = 0
    running_iou = 0

    for imgs, masks in train_loader:

        imgs = imgs.to(device)
        masks = masks.to(device)

        masks = masks.squeeze(2)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        dice = dice_score(outputs, masks)
        iou = iou_score(outputs, masks)

        running_dice += dice.item()
        running_iou += iou.item()

    avg_loss = running_loss / len(train_loader)
    avg_dice = running_dice / len(train_loader)
    avg_iou = running_iou / len(train_loader)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Loss: {avg_loss:.3f}")
    print(f"Dice Score: {avg_dice:.3f}")
    print(f"IoU: {avg_iou:.3f}")


Epoch [1/5]
Loss: 0.204
Dice Score: 0.280
IoU: 0.189

Epoch [2/5]
Loss: 0.077
Dice Score: 0.562
IoU: 0.412

Epoch [3/5]
Loss: 0.046
Dice Score: 0.640
IoU: 0.492

Epoch [4/5]
Loss: 0.034
Dice Score: 0.688
IoU: 0.545

Epoch [5/5]
Loss: 0.027
Dice Score: 0.730
IoU: 0.592


In [14]:
torch.save(model.state_dict(), "unetplusplus_baseline.pth")